In [ ]:
import pandas as pd
import scipy.stats as stats
import altair as alt
import numpy as np
from pathlib import Path
alt.data_transformers.disable_max_rows()

# Data Processing Helper Functions

In [ ]:
def spliceai_mapper(df):

    df['maxSpliceAI'] = df[['spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL']].max(axis = 1)

    df['splice_impact'] = '< Threshold'
    SPLICE_IMPACT_COLS = {
        'spliceAI_DS_AG': 'Acceptor Gain',
        'spliceAI_DS_AL': 'Acceptor Loss',
        'spliceAI_DS_DG': 'Donor Gain',
        'spliceAI_DS_DL': 'Donor Loss',
    }

    for col, label in SPLICE_IMPACT_COLS.items():
        df.loc[df[col] >= 0.2, 'splice_impact'] = label
    
    return df

In [ ]:
def molecular_consequence_mapper(df, remap_col):

    df = df.dropna(subset=[remap_col]).copy()
    CONSEQUENCE_EXACT = {
            'synonymous_variant': 'Synonymous',
            'intron_variant':     'Intron',
            'stop_gained':        'Stop Gained',
            'stop_lost':          'Stop Lost',
            'start_lost':         'Start Lost',
            'inframe_indel':      'Inframe Indel',
        }

    CONSEQUENCE_CONTAINS = {
        'missense': 'Missense',
        'site':     'Canonical Splice',
        'ing_var':  'Splice Region',
        'UTR':      'UTR Variant',
    }

    df[remap_col] = df[remap_col].replace(CONSEQUENCE_EXACT)
    for pattern, label in CONSEQUENCE_CONTAINS.items():
        df.loc[df[remap_col].str.contains(pattern), remap_col] = label
    

    return df

In [ ]:
raw_df = pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/20260424_CAVASGE_SangerSGE_FindlaySGE.xlsx')

raw_df = raw_df.loc[(raw_df['ref_allele'].str.len()==1) & (raw_df['alt_allele'].str.len()==1)].copy()

raw_df['pos_id']=raw_df['Gene']+':'+raw_df['hg38_start'].astype(str)+':'+raw_df['alt_allele']
raw_df['auth_reported_func_class'] = raw_df['auth_reported_func_class'].replace({
    'LOF':           'functionally_abnormal',
    'LOF1':          'functionally_abnormal',
    'LOF2':          'functionally_abnormal',
    'depleted':      'functionally_abnormal',
    'slow depleted': 'functionally_abnormal',
    'fast depleted': 'functionally_abnormal',
    'slow depleting':'functionally_abnormal',
    'fast depleting':'functionally_abnormal',
    'enriched':      'functionally_normal',
    'FUNC':          'functionally_normal',
    'Neutral':       'functionally_normal',
    'unchanged':     'functionally_normal',
    'INT':           'indeterminate',
    'Intermediate':  'indeterminate',
})

df = spliceai_mapper(raw_df)
df = molecular_consequence_mapper(df, 'simplified_consequence')
df.head()


In [ ]:
def rna_merge_prepper(df, gene=None, pos_col='pos', alt_col='alt', threshold=None):

    if gene is not None:
        df['pos_id'] = gene + ':' + df[pos_col].astype(str) + ':' + df[alt_col]
    else:
        df['pos_id'] = df['Gene'] + ':' + df[pos_col].astype(str) + ':' + df[alt_col]
    df = df.dropna(subset=['rna_score']).copy()

    df['rna_consequence'] = 'normal'
    df.loc[df['rna_score'] <= threshold, 'rna_consequence'] = 'low'

    keep_cols = ['pos_id', 'rna_score', 'rna_consequence']
    if 'maxSpliceAI' in df.columns:
        keep_cols.append('maxSpliceAI')

    return df[keep_cols]

In [ ]:
vhl_data = pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/2025_VHLBuckley.xlsx')
vhl_data = vhl_data.replace({'LOF1': 'LoF', 'LOF2': 'LoF',
                             'STOP_GAINED': 'Nonsense',
                             'NON_SYNONYMOUS': 'Missense',
                             'SYNONYMOUS': 'Synonymous',
                             'CANONICAL_SPLICE': 'Canonical Splice',
                             'INTRONIC': 'Intronic',
                             'SPLICE_SITE': 'Splice Region',
                             'STOP_LOST': 'Stop Lost'})
vhl_data = vhl_data.rename(columns={'max_spliceAI': 'maxSpliceAI'})

# Snapshot SpliceAI before rna_merge_prepper drops rna_score-NaN rows.
# VHL intron variants have maxSpliceAI populated but no rna_score, so they
# would otherwise be dropped before reaching the merge in the next cell.
vhl_data['pos_id'] = 'VHL:' + vhl_data['hg38_pos'].astype(str) + ':' + vhl_data['alt']
vhl_spliceai = vhl_data[['pos_id', 'maxSpliceAI']].dropna(subset=['maxSpliceAI']).copy()

vhl_data = rna_merge_prepper(vhl_data, gene='VHL', pos_col='hg38_pos', threshold=-3)
print(vhl_data)

In [ ]:
findlay_data=pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/20260422_BRCA1_Findlay2018_wRNA.xlsx')
findlay_data = molecular_consequence_mapper(findlay_data,'simplified_consequence')

findlay_data=findlay_data.rename(columns={
                                          'mean.rna.score': 'rna_score'}
                                          )

findlay_data = rna_merge_prepper(findlay_data, gene='BRCA1', pos_col='hg38_start', alt_col='alt_allele', threshold=-2)

print(findlay_data)

In [ ]:
cava_sge = Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna')

matches = list(cava_sge.glob("*allscores*"))

all_wrna = [vhl_data, findlay_data]

rna_threshold_dict = {
    'BARD1': -1.244,
    'RAD51D': -2.86
}
for match in matches:
    gene = str(match).split('/')[-1].split('.')[0]
    gene_df = pd.read_csv(match, sep='\t')

    if len(gene_df.dropna(subset=['RNA_score'])) == 0:
        continue

    gene_df = gene_df.rename(columns={'RNA_score': 'rna_score'})
    gene_df['Gene'] = gene_df['exon'].transform(lambda x: x.split('_')[0])

    rna_threshold = rna_threshold_dict[gene]
    wrna = rna_merge_prepper(gene_df, threshold=rna_threshold)

    all_wrna.append(wrna)

cava_w_rna = pd.concat(all_wrna)
print(cava_w_rna)

df = df.merge(
    cava_w_rna,
    on='pos_id',
    how='left',
    suffixes=('', '_new')
)

if 'maxSpliceAI_new' in df.columns:
    df['maxSpliceAI'] = df['maxSpliceAI'].fillna(df['maxSpliceAI_new'])
    df = df.drop(columns=['maxSpliceAI_new'])

# Fill remaining VHL maxSpliceAI gaps from the Buckley snapshot.
df = df.merge(vhl_spliceai, on='pos_id', how='left', suffixes=('', '_vhl'))
if 'maxSpliceAI_vhl' in df.columns:
    df['maxSpliceAI'] = df['maxSpliceAI'].fillna(df['maxSpliceAI_vhl'])
    df = df.drop(columns=['maxSpliceAI_vhl'])


# Intial Data Processing and Z-Score Normalization

In [ ]:
normal_mask = df['auth_reported_func_class'] == 'functionally_normal'

df['z_score'] = df.groupby('Gene')['auth_reported_score'].transform(
    lambda x: (x - x[normal_mask.reindex(x.index)].mean()) 
              / x[normal_mask.reindex(x.index)].std()
)

df = df[['Gene', 'auth_reported_score', 'auth_reported_func_class', 'z_score', 
         'simplified_consequence', 'maxSpliceAI', 'splice_impact', 'rna_score', 'rna_consequence']].dropna(subset=['auth_reported_func_class'])

## Z-score Normalization Sanity Check

In [ ]:
z_score_plot = alt.Chart(df).mark_boxplot().encode(
    x='Gene',
    y='z_score:Q'
).facet('auth_reported_func_class')

z_score_plot.display()

# Intron Variants, Z-score normalized

In [ ]:
# Scatter plot helper function

def corr_scatter(df, consequence, rep1, rep2, gene):
    rep1_max = df[rep1].max(axis = 0)
    rep2_max = df[rep2].max(axis = 0)
    rep2_min = df[rep2].min(axis = 0)

    x_max = rep1_max * 1.05
    y_max = rep2_max * 1.05
    y_min = rep2_min * 1.05

    df = df.dropna(subset = [rep1, rep2]).copy()

    df = df.loc[df['simplified_consequence']==consequence]


    scatter = alt.Chart(df).mark_circle().encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)
                  ),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        color='splice_impact:N',
        tooltip = ['Gene', 'auth_reported_score']
    )

    corr,_=stats.pearsonr(df[rep1], df[rep2])

    r_text = alt.Chart(pd.DataFrame({
        rep1: [x_max * 0.95],
        rep2: [1],
        'text': [f'r = {corr:.3f}']
    })).mark_text(
        align='right',
        baseline='bottom',
        fontSize=18,
        fontWeight='bold',
        color='black'
    ).encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        text='text:N'
    )

    scatter = (scatter+r_text).properties(title = gene).resolve_scale(x = 'shared', y = 'shared').display()

    rep_test = f'{rep1} vs. {rep2}'
    return scatter, gene, rep_test, corr

In [ ]:
def compute_correlations(df, rep1, rep2, consequences=None):
    # Per-gene Pearson r, then a median across genes for the "All (median)" row.

    work = df.dropna(subset=[rep1, rep2]).copy()

    if consequences is None:
        consequences = sorted(work['simplified_consequence'].dropna().unique())
    elif isinstance(consequences, str):
        consequences = [consequences]

    rows = []
    for cons in consequences:
        sub = work.loc[work['simplified_consequence'] == cons]
        if len(sub) < 3:
            continue
        gene_rs = []
        for gene, g in sub.groupby('Gene'):
            if len(g) < 3:
                continue
            if g[rep1].std() == 0 or g[rep2].std() == 0:
                continue  # constant input -> Pearson r is undefined
            r, _ = stats.pearsonr(g[rep1], g[rep2])
            rows.append({'consequence': cons, 'Gene': gene, 'r': r, 'n': len(g)})
            gene_rs.append(r)
        if gene_rs:
            rows.append({
                'consequence': cons,
                'Gene': 'All (median)',
                'r': float(np.median(gene_rs)),
                'n': len(gene_rs)  # number of genes contributing to the median
            })

    return pd.DataFrame(rows)



def corr_heatmap(corr_df, rep1, rep2):
    row_order = list(corr_df['consequence'].unique())
    gene_order = ['All (median)', *corr_df.loc[corr_df['Gene'] != 'All (median)', 'Gene'].unique()]

    base = alt.Chart(corr_df).encode(
        x=alt.X('Gene:N', sort=gene_order),
        y=alt.Y('consequence:N', sort=row_order),
    )

    # Use white text on dark cells (high positive r > 0.5 or strong negative r < -0.6)
    text_color = alt.condition(
        'datum.r > 0.5 || datum.r < -0.6',
        alt.value('white'),
        alt.value('black')
    )

    heatmap = base.mark_rect().encode(
        color=alt.Color('r:Q',
            scale=alt.Scale(scheme='redblue', domainMid=0, domain=[-1, 1]),
            legend=alt.Legend(title='Pearson r')
        ),
        tooltip=['Gene', 'consequence', alt.Tooltip('r:Q', format='.3f'), alt.Tooltip('n:Q', title='n')]
    )

    r_text = base.mark_text(fontSize=12, dy=-5).encode(
        text=alt.Text('r:Q', format='.2f'),
        color=text_color
    )

    # n=variants for per-gene cells; suppressed for the median cell
    n_text = base.mark_text(fontSize=10, dy=7, opacity=0.85).transform_calculate(
        n_label='datum.Gene === "All (median)" ? "" : "n=" + datum.n'
    ).encode(
        text=alt.Text('n_label:N'),
        color=text_color
    )

    return (heatmap + r_text + n_text).properties(title=f'Pearson r: Fitness Score vs. Max SpliceAI', width=500, height = 400)


In [ ]:
scatter, _, _, _ = corr_scatter(df, 'Intron', 'z_score', 'maxSpliceAI', 'All SGE Intron Variants')

In [ ]:
# (label, filtered_df, allowed_consequences or None for all)

consequences = ['Intron', 'Splice Region', 'Missense', 'Synonymous', 'Canonical Splice']

subsets = [
    ('',        df,                                                                  None),
    ('LoF',     df.loc[df['auth_reported_func_class'] == 'functionally_abnormal'],   None),
    ('LoF Low RNA', df.loc[(df['rna_consequence'] == 'low') & (df['auth_reported_func_class']=='functionally_abnormal')],['Missense', 'Synonymous'])
]

corr_df = pd.concat([
    compute_correlations(subset, 'auth_reported_score', 'maxSpliceAI', consequences=[cons])
      .assign(consequence=f'{label} {cons}'.strip())
    for cons in consequences
    for label, subset, allowed_cons in subsets
    if allowed_cons is None or cons in allowed_cons
], ignore_index=True)


In [ ]:
splice_ai_corr_map = corr_heatmap(corr_df, 'z_score', 'maxSpliceAI')

splice_ai_corr_map.display()
splice_ai_corr_map.save('/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/20260427_FitnessScore_vs_SpliceAI_PearsonR.png', dpi=600)


In [ ]:
def compute_spliceai_coverage(df, threshold=0.2, consequences=None):
    work = df.dropna(subset=['maxSpliceAI']).copy()

    if consequences is None:
        consequences = sorted(work['simplified_consequence'].dropna().unique())
    elif isinstance(consequences, str):
        consequences = [consequences]

    rows = []
    for cons in consequences:
        sub = work.loc[work['simplified_consequence'] == cons]
        if len(sub) == 0:
            continue
        for gene, g in sub.groupby('Gene'):
            rows.append({
                'consequence': cons, 'Gene': gene,
                'pct': (g['maxSpliceAI'] >= threshold).mean() * 100,
                'n': len(g)
            })
        rows.append({
            'consequence': cons, 'Gene': 'All',
            'pct': (sub['maxSpliceAI'] >= threshold).mean() * 100,
            'n': len(sub)
        })

    return pd.DataFrame(rows)


def coverage_heatmap(cov_df, threshold, var_type):
    row_order = list(cov_df['consequence'].unique())

    base = alt.Chart(cov_df).encode(
        x=alt.X('Gene:N', sort=['All', *cov_df.loc[cov_df['Gene'] != 'All', 'Gene'].unique()]),
        y=alt.Y('consequence:N', sort=row_order),
    )

    heatmap = base.mark_rect().encode(
        color=alt.Color('pct:Q',
            scale=alt.Scale(scheme='oranges', domain=[0, 100]),
            legend=alt.Legend(title='% predicted')
        ),
        tooltip=['Gene', 'consequence',
                 alt.Tooltip('pct:Q', format='.1f', title='% predicted'),
                 alt.Tooltip('n:Q', title='n')]
    )

    pct_text = base.mark_text(fontSize=12, dy=-5).encode(
        text=alt.Text('pct:Q', format='.1f'),
        color=alt.condition(
            alt.datum.pct > 60, alt.value('white'), alt.value('black')
        )
    )

    n_text = base.mark_text(fontSize=10, dy=7, opacity=0.85).transform_calculate(
        n_label='"n=" + datum.n'
    ).encode(
        text=alt.Text('n_label:N'),
        color=alt.condition(
            alt.datum.pct > 60, alt.value('white'), alt.value('black')
        )
    )

    return (heatmap + pct_text + n_text).properties(
        title=f'% {var_type} variants with maxSpliceAI ≥ {threshold}', width=500, height = 300
    )


In [ ]:
consequences = ['Intron', 'Splice Region', 'Missense', 'Synonymous', 'Canonical Splice']

subsets = [
    ('LoF',     df.loc[df['auth_reported_func_class'] == 'functionally_abnormal'], None),
    ('Low RNA LoF', df.loc[(df['rna_consequence'] == 'low') & (df['auth_reported_func_class'] == 'functionally_abnormal')],['Missense', 'Synonymous', 'Splice Region']),
]

cov_df = pd.concat([
    compute_spliceai_coverage(subset, threshold=0.2, consequences=[cons])
      .assign(consequence=f'{label} {cons}'.strip())
    for cons in consequences
    for label, subset, allowed_cons in subsets
    if allowed_cons is None or cons in allowed_cons
], ignore_index=True)

lof_spliceai_map = coverage_heatmap(cov_df, threshold=0.2, var_type = 'LoF')

lof_spliceai_map.display()
lof_spliceai_map.save('/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/20260427_ProportionLOF_wSpliceAIHit.png', dpi=600)


In [ ]:
consequences = ['Intron', 'Splice Region', 'Missense', 'Synonymous', 'Canonical Splice']

subsets = [
    ('Normal',            df.loc[df['auth_reported_func_class'] == 'functionally_normal'], None),
    ('Normal RNA Normal', df.loc[(df['rna_consequence'] == 'normal') & (df['auth_reported_func_class'] == 'functionally_normal')], ['Missense', 'Synonymous', 'Splice Region']),
]

cov_df = pd.concat([
    compute_spliceai_coverage(subset, threshold=0.2, consequences=[cons])
      .assign(consequence=f'{label} {cons}'.strip())
    for cons in consequences
    for label, subset, allowed_cons in subsets
    if allowed_cons is None or cons in allowed_cons
], ignore_index=True)

normal_splice_ai_map = coverage_heatmap(cov_df, threshold=0.2, var_type='Normal')

normal_splice_ai_map.display()
normal_splice_ai_map.save('/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/20260427_ProportionNormal_wSpliceAIHit.png', dpi=600)

In [ ]:
#Single gene pearson r scatter
def gene_scatter(df, gene, consequence, rep1='auth_reported_score', rep2='maxSpliceAI',
                 lof_only=False, low_rna_only=False):
    work = (df.loc[(df['Gene'] == gene) & (df['simplified_consequence'] == consequence)]
              .dropna(subset=[rep1, rep2])
              .copy())

    if lof_only:
        work = work.loc[work['auth_reported_func_class'] == 'functionally_abnormal']
    if low_rna_only:
        work = work.loc[work['rna_consequence'] == 'low']

    if len(work) < 3:
        raise ValueError(f'Not enough data for {gene} / {consequence} (n={len(work)})')



    r, _ = stats.pearsonr(work[rep1], work[rep2])

    x_pad = (work[rep1].max() - work[rep1].min()) * 0.05
    y_pad = (work[rep2].max() - work[rep2].min()) * 0.05
    x_domain = [work[rep1].min() - x_pad, work[rep1].max() + x_pad]
    y_domain = [work[rep2].min() - y_pad, work[rep2].max() + y_pad]

    base = alt.Chart(work)

    scatter = base.mark_circle(opacity=0.7).encode(
        x=alt.X(f'{rep1}:Q', scale=alt.Scale(domain=x_domain)),
        y=alt.Y(f'{rep2}:Q', scale=alt.Scale(domain=y_domain)),
        color='auth_reported_func_class:N',
        tooltip=['Gene', rep1, rep2, 'auth_reported_func_class', 'splice_impact']
    )

    trendline = base.transform_regression(rep1, rep2).mark_line(
        color='black', strokeDash=[4, 2], size=1.5
    ).encode(
        x=alt.X(f'{rep1}:Q'),
        y=alt.Y(f'{rep2}:Q')
    )

    r_label = alt.Chart(pd.DataFrame({
        rep1:   [x_domain[1]],
        rep2:   [y_domain[1]],
        'text': [f'r = {r:.3f}  n = {len(work)}']
    })).mark_text(
        align='right', baseline='top', fontSize=13, fontWeight='bold', color='black'
    ).encode(
        x=alt.X(f'{rep1}:Q'),
        y=alt.Y(f'{rep2}:Q'),
        text='text:N'
    )

    return (scatter + trendline + r_label).properties(
        title=f'{gene} — {consequence}', width=300, height=300
    )


In [ ]:
gene_scatter(df, 'BARD1', 'Synonymous', lof_only=False).display()

In [ ]:
scatter, _, _, _ = corr_scatter(df, 'Missense', 'z_score', 'maxSpliceAI', 'All SGE Intron Variants')